# Analyze feature contributions in held-out test set -- compare to exploratory analyses in train set

- Salience
- LMG Feature Importance

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import (
    BEST_PARAM_DICT,
    get_test_df_path,
    get_perm_seed,
    col_config_sensitivity,
)
from src.sig import prep_train_test_model
from src.residual_func import get_saliency_map
from src.lmg_imp import get_confirm_lmg


import joblib
from functools import reduce
import pandas as pd

model_dir = BASE_PATH / "models"
df_dir = BASE_PATH / "data/test_data_toy"

## Salience

In [ ]:
for impact_type in ["self", "parent"]:
    model_params = BEST_PARAM_DICT["SQT"][impact_type]
    model_SQT = joblib.load(model_dir / f"SQT_{impact_type}.joblib")
    coef_df = model_SQT.get_coef_df()
    coef_df["stage_feat"] = coef_df["stage"] + "_" + coef_df["feature"]
    coef_df = coef_df.drop(["stage", "feature"], axis=1)
    coef_df = coef_df.set_index("stage_feat")
    coef_df = coef_df.rename(columns={"coef": f"coef_{impact_type}"})
    test_df_path = get_test_df_path(
        df_dir,
        llm_model=model_params["llm_model"],
        agg_type=model_params["agg_type"],
        desc_seg_type=model_params["desc_seg_type"],
        impact_type=impact_type,
        temp=model_params["temp"],
    )
    test_df = pd.read_parquet(test_df_path)
    model_params["impact_type"] = impact_type
    summary = get_saliency_map(
        config_dict=model_params,
        model=model_SQT,
        df=test_df,
        coef_df=coef_df,
    )
    display(summary)

## LMG

In [ ]:
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data_toy"
model_dir = BASE_PATH / "models"
all_perm_dir = all_perm_dir = BASE_PATH / "results/perm_res/preds"
START_PERM = 0
END_PERM = 5
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}

In [ ]:
res_dict = {}
for impact_type in ["parent", "self"]:
    print(f"{impact_type}...")
    model_params = BEST_PARAM_DICT["SQT"][impact_type]
    X_train, y_train, X_test, y_test, model = prep_train_test_model(
        selected_params=model_params,
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
        shared_dict=SHARED_DICT,
    )
    model_config_dict = {
        **{
            k: v
            for k, v in model_params.items()
            if k not in ["two_layer_base", "outcome_sub_list", "llm_model", "temp"]
        },
        "quest_cols": col_config_sensitivity[f"{impact_type}_quest"],
        "impact_type": impact_type,
        **SHARED_DICT,
    }

    final_df = get_confirm_lmg(
        perm_dir=all_perm_dir,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        model_config_dict=model_config_dict,
        start_perm=START_PERM,
        end_perm=END_PERM,
    )
    res_dict[impact_type] = final_df
    display(final_df)